In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm

ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x12094f680>, default_metadata=(), model_kwargs={})

In [4]:
llm.invoke("How will the weather be in munich today?")

AIMessage(content="To give you the most accurate, real-time weather information for Munich today, I'd recommend checking a live weather service. However, based on typical forecasts:\n\n*   **Temperature:** Expect a high around **22-25°C (72-77°F)** and a low overnight around **10-13°C (50-55°F)**.\n*   **Sky Conditions:** It looks like it will be **partly cloudy to mostly sunny** for much of the day.\n*   **Precipitation:** There's a **low chance of rain**, generally staying **dry**.\n*   **Wind:** Winds will be **light to moderate**.\n\nOverall, it should be a **pleasant and mild day** in Munich.\n\nFor the most up-to-the-minute details, please check a reliable weather app or website like AccuWeather, The Weather Channel, or your local meteorological service (e.g., DWD in Germany).", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cbda9-cedc-71e0-9456-3387207ce45

In [5]:
from langchain_core.tools import tool

In [6]:
@tool
def get_weather(location: str):
    """Call to get the current weather."""
    if location.lower() in ["munich"]:
        return "It's 15 degrees Celsius and cloudy."
    else:
        return "It's 32 degrees Celsius and sunny."

@tool
def check_seating_availability(location: str, seating_type: str):
    """Call to check seating availability."""
    if location.lower() == "munich" and seating_type.lower() == "outdoor":
        return "Yes, we still have seats available outdoors."
    elif location.lower() == "munich" and seating_type.lower() == "indoor":
        return "Yes, we have indoor seating available."
    else:
        return "Sorry, seating information for this location is unavailable."


tools = [get_weather, check_seating_availability] 

In [7]:
llm_with_tools = llm.bind_tools(tools)
llm_with_tools

RunnableBinding(bound=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x12094f680>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Call to get the current weather.', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'check_seating_availability', 'description': 'Call to check seating availability.', 'parameters

In [8]:
result = llm_with_tools.invoke("How will the weather be in munich today?")
print(result)

content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "munich"}'}, '__gemini_function_call_thought_signatures__': {'bbb40e98-96aa-4b6c-8bf8-f8ab0c79be59': 'CpYCAb4+9vve8iNqRHwbCUyOlgVl6ZhP52bL4cYjXHRPGano5XNfkkR3fjKIqf6uCNihTYZaboPokjMXS1rfR4gsbuK2BoWUnqJ6QafYxWOC7zPT23/JmIhrnn7mdE0bLn56F4IA3tgeBYeTjPqDhK27a/EYIzFH2xBFw7TMeFQZLgiNny1SbzGzAewCl4tXgudtB4JBcTgiJL2AOhVP++zfgTIs3tPqMvy51am3eECUvt/iEQuA+yv6beU8uukeY+BWVbeXcL1D7H501MC5c8AtMmeb36tNIQ2gNIMqI3irHulruEnimE+z+agqiwIQQ6ECa7FvLyX3ya12h4oeA1k4dkW3KmKGZleY0Gj6WuvzodgmFcJDZrk='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019cbda9-f0b4-72a3-ac20-276b5fbdd0e8-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'munich'}, 'id': 'bbb40e98-96aa-4b6c-8bf8-f8ab0c79be59', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 108, 'output_tokens': 78, 'total_tokens': 18

In [9]:
result.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'munich'},
  'id': 'bbb40e98-96aa-4b6c-8bf8-f8ab0c79be59',
  'type': 'tool_call'}]

In [12]:
result = llm_with_tools.invoke(
    "How will the weather be in munich today? Do you still have seats outdoor available?"
)
result

AIMessage(content='', additional_kwargs={'function_call': {'name': 'check_seating_availability', 'arguments': '{"seating_type": "outdoor", "location": "munich"}'}, '__gemini_function_call_thought_signatures__': {'7b519375-8197-459c-a4f4-57e83248843c': 'CpcDAb4+9vs9n0SP+0pmW+7GS2PT0OJYFS5iA3VLHdlVKdaI+1pp/G9Xb63Bg4csIp1XZHkd4gRV7m0WYBrGQDuXZFxrhjtBPteGlQpuPVo68xF/lVbyQXjWE7ee+L8AX0Js5hsNmER6FE3P+8sYvim+BEUSg8o4UNnuo8ZnP71WZNNA9Z7XjAIL4/n996U6LO5h4q7LR5+NT2PtnS6wcn7UJhBz43ZE/iWzYux3wcG3jRzOlV9r0KsXT1gs2vskFyaLJSjPt07vK69FEZP+K3PJiuBxa6/63cQfTekcY5zWdT99C584gN9zNxlgYdFFp3pOXslrm2y/buPVJRMvGvrWDxwRvOdPG1w6SAna2lYyL+QraDPuSjujfDIQaOCNXOIdDvowVmdxoWwbRADbRFhQXi5mHHKQIOu4tzh6By8AcHLqXLFmlnkPSjJo5sNf8/GJUR0klQwZHUTgF+Aex2h1euLEJsWdQ8Q0s0pHuMtRyP3+MEqyJsSmUlxBMEeOJ+eRHF0pMJEqqLkE7Cp6Jsid1olKcyD+uRU='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cbdb2-4752-7aa1-8216-75f7d6c285d3-0', tool_c

In [13]:
result.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'munich'},
  'id': '7b519375-8197-459c-a4f4-57e83248843c',
  'type': 'tool_call'},
 {'name': 'check_seating_availability',
  'args': {'seating_type': 'outdoor', 'location': 'munich'},
  'id': '9799bc57-ddd4-46ed-ace7-5c3b23f73379',
  'type': 'tool_call'}]

In [14]:
from langchain_core.messages import HumanMessage, ToolMessage

In [15]:
messages = [
    HumanMessage(
        "How will the weather be in munich today? Do you still have seats outdoor available?"
    )
]

llm_output = llm_with_tools.invoke(messages)
llm_output

AIMessage(content='', additional_kwargs={'function_call': {'name': 'check_seating_availability', 'arguments': '{"location": "munich", "seating_type": "outdoor"}'}, '__gemini_function_call_thought_signatures__': {'768747eb-5f04-4491-8dda-3eed9f8a84e9': 'CogEAb4+9vtbFjrB5ZIhGcjWlIInZWgV1r4jA9Ohog5IU2m6cyq6hDICBXEtU+l4iJz8CDrcGZtEalxT7sN0gIdFCcF9eAMiy9A1+mhkqZa7DFxRof/9pJb7nPTQ8xEKOwoNuEOqTbJox3Wbwf494xj0O4djahNy3IQQvnHBLlE02maq5BXx32Dj6XpGHNfH1u3/7Qe23Sv0k24UzGRF6RSHcyPh32EHvv96iZ7y4BkqaV9yheRvyarKgQf+vTBfGHhF9rxm43rLunTQAevSZuXTj7eL4cor5GIoo4T2n8zYznRBDa+3m2+X+eo+VuZK1Ho9J/KFPbtHpPSsjfJBhUlq5YBs+iP2wZpTy+actj9vpAm6/jwS3IrBmyLB+ySaFnWh4YpI9i4CJ3Ydj4eRnEe+IooyKLAEOZ9j4T9O6Sctv9D3V0a/xHKF8Zc6ZikKbOerqyfOrA0CsdODPoxCaH72vg1fSW8LvjMUou+ason/yO3Tn5mn5gb5ZeRfXW01vLrdu3ucC7h4lXz9grhPZTiCOITrVVq4ECatnxne31WAxLuOipP+mKSmfqUab+LoEjxZZoN2Gdat/3z8OgQepYs3uXNi3KQScM1rDGMhqy+QaTGqyFvWMTxqYmp/UM95EdI9/danLPCeesGvAbAB8WmTQTS56MxNtAjdyCT9aS/lxaRiGBaDKoryLA=='}}, response_metadata={'finish_reason': 'STOP'

In [17]:
messages.append(llm_output)
print(messages)

[HumanMessage(content='How will the weather be in munich today? Do you still have seats outdoor available?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'function_call': {'name': 'check_seating_availability', 'arguments': '{"location": "munich", "seating_type": "outdoor"}'}, '__gemini_function_call_thought_signatures__': {'768747eb-5f04-4491-8dda-3eed9f8a84e9': 'CogEAb4+9vtbFjrB5ZIhGcjWlIInZWgV1r4jA9Ohog5IU2m6cyq6hDICBXEtU+l4iJz8CDrcGZtEalxT7sN0gIdFCcF9eAMiy9A1+mhkqZa7DFxRof/9pJb7nPTQ8xEKOwoNuEOqTbJox3Wbwf494xj0O4djahNy3IQQvnHBLlE02maq5BXx32Dj6XpGHNfH1u3/7Qe23Sv0k24UzGRF6RSHcyPh32EHvv96iZ7y4BkqaV9yheRvyarKgQf+vTBfGHhF9rxm43rLunTQAevSZuXTj7eL4cor5GIoo4T2n8zYznRBDa+3m2+X+eo+VuZK1Ho9J/KFPbtHpPSsjfJBhUlq5YBs+iP2wZpTy+actj9vpAm6/jwS3IrBmyLB+ySaFnWh4YpI9i4CJ3Ydj4eRnEe+IooyKLAEOZ9j4T9O6Sctv9D3V0a/xHKF8Zc6ZikKbOerqyfOrA0CsdODPoxCaH72vg1fSW8LvjMUou+ason/yO3Tn5mn5gb5ZeRfXW01vLrdu3ucC7h4lXz9grhPZTiCOITrVVq4ECatnxne31WAxLuOipP+mKSmfqUab+LoEjxZZoN2Gdat/3z8O

In [18]:
tool_mapping = {
    "get_weather": get_weather,
    "check_seating_availability": check_seating_availability,
}
tool_mapping

{'get_weather': StructuredTool(name='get_weather', description='Call to get the current weather.', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x110774540>),
 'check_seating_availability': StructuredTool(name='check_seating_availability', description='Call to check seating availability.', args_schema=<class 'langchain_core.utils.pydantic.check_seating_availability'>, func=<function check_seating_availability at 0x121031620>)}

In [19]:
llm_output.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'munich'},
  'id': '768747eb-5f04-4491-8dda-3eed9f8a84e9',
  'type': 'tool_call'},
 {'name': 'check_seating_availability',
  'args': {'location': 'munich', 'seating_type': 'outdoor'},
  'id': 'b7bc3933-0fa2-4a2b-bd6c-d8a3e0e2068b',
  'type': 'tool_call'}]

In [20]:
for tool_call in llm_output.tool_calls:
    tool = tool_mapping[tool_call["name"].lower()]
    tool_output = tool.invoke(tool_call["args"])
    messages.append(ToolMessage(tool_output, tool_call_id=tool_call["id"]))

In [21]:
messages

[HumanMessage(content='How will the weather be in munich today? Do you still have seats outdoor available?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'check_seating_availability', 'arguments': '{"location": "munich", "seating_type": "outdoor"}'}, '__gemini_function_call_thought_signatures__': {'768747eb-5f04-4491-8dda-3eed9f8a84e9': 'CogEAb4+9vtbFjrB5ZIhGcjWlIInZWgV1r4jA9Ohog5IU2m6cyq6hDICBXEtU+l4iJz8CDrcGZtEalxT7sN0gIdFCcF9eAMiy9A1+mhkqZa7DFxRof/9pJb7nPTQ8xEKOwoNuEOqTbJox3Wbwf494xj0O4djahNy3IQQvnHBLlE02maq5BXx32Dj6XpGHNfH1u3/7Qe23Sv0k24UzGRF6RSHcyPh32EHvv96iZ7y4BkqaV9yheRvyarKgQf+vTBfGHhF9rxm43rLunTQAevSZuXTj7eL4cor5GIoo4T2n8zYznRBDa+3m2+X+eo+VuZK1Ho9J/KFPbtHpPSsjfJBhUlq5YBs+iP2wZpTy+actj9vpAm6/jwS3IrBmyLB+ySaFnWh4YpI9i4CJ3Ydj4eRnEe+IooyKLAEOZ9j4T9O6Sctv9D3V0a/xHKF8Zc6ZikKbOerqyfOrA0CsdODPoxCaH72vg1fSW8LvjMUou+ason/yO3Tn5mn5gb5ZeRfXW01vLrdu3ucC7h4lXz9grhPZTiCOITrVVq4ECatnxne31WAxLuOipP+mKSmfqUab+LoEjxZZoN2Gdat/3z8

In [22]:
llm_with_tools.invoke(messages)

AIMessage(content=[{'type': 'text', 'text': "It's 15 degrees Celsius and cloudy in Munich today. Yes, we still have seats available outdoors.", 'extras': {'signature': 'CqkDAb4+9vszAIPnZWUlSWDBMrRHST907SzoG86A1Wotx/pyeIX8cYlNjXMK/auk/oSbcliSGLjJm2/gFnDir8DaXB+85AOZWfU8cXUvwfwAuEwdJm1z6B6Fv4Sc4xOl+9VutAmIcOFtNoECsQTHZu9NC3GYikdGvnLvfGM5h9fVLWeFwUJO5QD0jOT0VhtjyhMDGaJAo+y/h4rLguzJp0fW04RVw/FVyJJ4PWaSzpocxO7qLw07cfzcRBJLdxS2vKTYzMpX70sUkK0rHn1j9uDWwtS/YfeUjKI1wDBwbJMxO7I8Xc3mpDH3TUmvP2OYvcx84ax47gdepcCBAYi+C2Fd0l61Z8wj6fLawa2EPt7OgSShA2NXQhK/7g4lImC3cLHnyolcXvvsYvuvMeT3CXms2mRhjCv0BoZWd8sZ+fnE/bPskaQQql/GZJqWPjjlbSrqweTVSjWALNIeCwf8svC2QTLW2RntbUvVI1qqeOr9oEXkOLmc1VLzgULy/KTGNjDuISzmC0I+hGCdEfSnxCoKXzrZ/nn4xC8cIxWv3JuPljLjoDMwXN7PapQ='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cbdba-b41f-77b3-9b23-e65b5042b657-0', tool_calls=[], invalid_tool_calls=[], usage